# P07 — Auditoría end-to-end de un sistema de IA

**Caso:** Velora Finance

Velora quiere pasar de una colección de pruebas técnicas a una decisión formal sobre despliegue.

El sistema combina evidencias procedentes de los módulos anteriores:

- modelo de riesgo y threshold;
- fairness;
- Differential Privacy;
- explicabilidad;
- RAG;
- agente con tools y approval gate;
- logging y controles operativos.

En esta práctica construiremos un **audit gate reproducible**.

El objetivo no es “marcar casillas”, sino poder responder:

> **¿Qué evidencia sostiene la decisión de desplegar, condicionar o bloquear el sistema?**

Trabajaremos con:

- evidence pack;
- integridad mediante SHA-256;
- trazabilidad control → evidencia;
- reglas de control explícitas;
- findings;
- risk register;
- remediaciones;
- decisión `APPROVED / CONDITIONAL / BLOCKED`;
- generación automática de un informe de auditoría.

El dossier es sintético y está diseñado para integrar P01–P06.


## 1. Marco de referencia

La práctica utiliza un catálogo interno inspirado en tres referencias:

### NIST AI RMF

Organiza la gestión del riesgo en:

- **Govern**
- **Map**
- **Measure**
- **Manage**

### ISO/IEC 42001

Aporta la perspectiva de **sistema de gestión**, responsabilidades, procesos y mejora continua.

### EU AI Act

Usamos como referencia temas habituales de sistemas de alto riesgo:

- gestión de riesgos;
- calidad y gobernanza de datos;
- logging y trazabilidad;
- documentación;
- supervisión humana;
- precisión, robustez y ciberseguridad.

**Importante**

Este laboratorio:

- no determina jurídicamente si Velora es un sistema de alto riesgo;
- no certifica ISO/IEC 42001;
- no demuestra cumplimiento legal.

El objetivo es practicar **evidence-based governance**.

A 31 de octubre de 2026, las obligaciones de transparencia del AI Act ya son aplicables, mientras que la aplicación de las reglas de alto riesgo de Annex III está prevista, según el calendario vigente de la Comisión, para diciembre de 2027.


# 2. Instalar dependencias

Solo necesitamos librerías ligeras.

`jsonschema` se utiliza para validar la estructura del evidence pack.

No introducimos una plataforma GRC completa porque queremos ver de forma transparente cómo se conectan:

```text
control
  ↓
evidencia
  ↓
regla
  ↓
finding
  ↓
riesgo
  ↓
decisión
```


In [ ]:
!pip -q install jsonschema


In [ ]:
import json
import hashlib
import math
from copy import deepcopy
from datetime import datetime, timezone

import pandas as pd
from jsonschema import validate, ValidationError

pd.set_option("display.max_colwidth", 120)

print("Entorno preparado.")


# 3. Contexto del sistema

Antes de auditar métricas debemos saber **qué sistema estamos auditando**.

Documentamos:

- propósito;
- usuarios;
- personas afectadas;
- decisiones;
- componentes;
- owner;
- etapa del lifecycle.

Una métrica sin contexto no permite decidir si el riesgo es aceptable.


In [ ]:
SYSTEM_CONTEXT = {
    "system_id": "VELORA-AI-001",
    "name": "Velora Credit Decision Support",
    "version": "0.9.0-rc1",
    "stage": "PRE_PRODUCTION",
    "intended_use": (
        "Apoyar la evaluación de solicitudes de crédito y operaciones "
        "internas relacionadas, con supervisión humana."
    ),
    "out_of_scope": [
        "Aprobación automática de excepciones comerciales",
        "Modificación de límites sin autorización",
        "Decisiones basadas exclusivamente en contenido generado por LLM",
    ],
    "affected_people": [
        "Solicitantes de crédito",
        "Clientes existentes",
    ],
    "business_owner": "Credit Risk",
    "technical_owner": "AI Platform",
    "risk_owner": "Enterprise Risk",
    "components": [
        "Modelo tabular de riesgo",
        "Fairness mitigation",
        "DP-SGD experiment",
        "XAI",
        "RAG de políticas",
        "Agente con tools",
    ],
}

display(pd.Series(SYSTEM_CONTEXT).to_frame("valor"))


## 4. Definir el evidence pack

Cada evidencia tendrá:

- identificador;
- origen;
- fecha;
- versión;
- tipo;
- contenido.

La evidencia puede ser:

- métrica;
- resultado booleano;
- documento;
- log;
- configuración.

En un sistema real estos elementos podrían vivir en MLflow, un data catalog, un SIEM, un GRC o un repositorio documental.

Aquí los unificamos en un único bundle para poder auditarlo.


In [ ]:
EVIDENCE_SCHEMA = {
    "type": "object",
    "required": [
        "evidence_id",
        "source",
        "timestamp",
        "version",
        "type",
        "payload",
    ],
    "properties": {
        "evidence_id": {"type": "string"},
        "source": {"type": "string"},
        "timestamp": {"type": "string"},
        "version": {"type": "string"},
        "type": {"type": "string"},
        "payload": {"type": "object"},
    },
    "additionalProperties": False,
}

AUDIT_TIMESTAMP = "2026-10-31T18:00:00Z"


# 5. Evidencias de P01 — rendimiento y threshold

Usamos el snapshot validado en P01.

El modelo a threshold `0.5` tiene:

- accuracy ≈ 0.897;
- recall de impago ≈ 0.214;
- 55 falsos negativos sobre 70 impagos reales.

Para gobernanza, el dato crítico no es que “accuracy sea alta”.

La pregunta es:

> ¿El comportamiento satisface los límites definidos para el uso previsto?

También incluimos varias alternativas de threshold ya evaluadas.


In [ ]:
THRESHOLD_OPTIONS = {
    0.2: {
        "accuracy": 0.822,
        "precision": 0.350,
        "recall": 0.614,
        "fp": 80,
        "fn": 27,
    },
    0.3: {
        "accuracy": 0.872,
        "precision": 0.449,
        "recall": 0.443,
        "fp": 38,
        "fn": 39,
    },
    0.4: {
        "accuracy": 0.885,
        "precision": 0.514,
        "recall": 0.271,
        "fp": 18,
        "fn": 51,
    },
    0.5: {
        "accuracy": 0.897,
        "precision": 0.682,
        "recall": 0.214,
        "fp": 7,
        "fn": 55,
    },
    0.6: {
        "accuracy": 0.900,
        "precision": 0.917,
        "recall": 0.157,
        "fp": 1,
        "fn": 59,
    },
}

display(
    pd.DataFrame.from_dict(
        THRESHOLD_OPTIONS,
        orient="index",
    ).rename_axis("threshold")
)


# 6. Evidencias P02–P06

El audit pack contiene snapshots normalizados.

Los valores están preparados para el ejercicio de gobierno:

- no son una certificación;
- no sustituyen los notebooks originales;
- permiten evaluar de forma integrada controles heterogéneos.

En un proceso real, estos valores deberían ser referencias inmutables a artefactos producidos por pipelines o sistemas de observabilidad.


In [ ]:
def evidence(evidence_id, source, version, type_, payload):
    return {
        "evidence_id": evidence_id,
        "source": source,
        "timestamp": AUDIT_TIMESTAMP,
        "version": version,
        "type": type_,
        "payload": payload,
    }

BASE_EVIDENCE = [
    evidence(
        "E-P01-PERF",
        "P01",
        "1.0",
        "metric_snapshot",
        {
            "roc_auc": 0.824,
            "average_precision": 0.479,
            "selected_threshold": 0.5,
        },
    ),
    evidence(
        "E-P02-FAIR-BASE",
        "P02",
        "1.0",
        "metric_snapshot",
        {
            "demographic_parity_difference": 0.17,
            "equal_opportunity_difference": 0.14,
            "mitigated": False,
        },
    ),
    evidence(
        "E-P02-FAIR-MIT",
        "P02",
        "1.0",
        "metric_snapshot",
        {
            "demographic_parity_difference": 0.08,
            "equal_opportunity_difference": 0.07,
            "mitigated": True,
        },
    ),
    evidence(
        "E-P03-PRIV",
        "P03",
        "1.0",
        "privacy_evidence",
        {
            "epsilon": 3.0,
            "delta": 1e-5,
            "accountant": "PRV",
            "privacy_unit_documented": True,
            "composition_tracked": True,
        },
    ),
    evidence(
        "E-P04-XAI",
        "P04",
        "1.0",
        "explainability_evidence",
        {
            "global_shap": True,
            "local_shap": True,
            "lime_fidelity_checked": True,
            "lime_stability_checked": True,
            "counterfactual_constraints": True,
            "causal_limit_documented": True,
        },
    ),
    evidence(
        "E-P05-RAG",
        "P05",
        "1.0",
        "rag_evidence",
        {
            "retrieval_evaluated": True,
            "hit_at_k": 1.0,
            "mrr": 0.875,
            "invalid_citations": 0,
            "unsupported_question_abstained": True,
            "indirect_injection_tested": True,
            "injection_control_effective": True,
        },
    ),
    evidence(
        "E-P06-AGENT",
        "P06",
        "1.0",
        "agent_evidence",
        {
            "least_privilege_scope": True,
            "tool_calls_structured": True,
            "backend_enforces_approval": True,
            "approval_outside_agent": True,
            "approval_single_use": True,
            "replay_denied": True,
            "tool_response_injection_tested": True,
            "response_id_logged": True,
            "call_id_logged": True,
        },
    ),
    evidence(
        "E-GOV-OWNERSHIP",
        "Governance",
        "0.9",
        "document",
        {
            "business_owner_assigned": True,
            "technical_owner_assigned": True,
            "risk_owner_assigned": True,
            "intended_use_documented": True,
            "out_of_scope_documented": True,
        },
    ),
    evidence(
        "E-OPS-MONITOR",
        "Operations",
        "0.5",
        "operational_control",
        {
            "monitoring_owner": True,
            "performance_drift_thresholds": False,
            "fairness_drift_thresholds": False,
            "rag_quality_monitoring": True,
            "agent_denial_rate_monitoring": True,
        },
    ),
    evidence(
        "E-OPS-INCIDENT",
        "Operations",
        "0.2",
        "operational_control",
        {
            "ai_incident_runbook": False,
            "escalation_owner": True,
            "rollback_procedure": False,
        },
    ),
]

print("Evidencias base:", len(BASE_EVIDENCE))


# 7. Validar esquema del evidence pack

Antes de evaluar controles comprobamos que todas las evidencias tienen la estructura esperada.

Esto no valida su veracidad.

Valida que el paquete sea **procesable y consistente**.

En auditoría técnica es útil separar:

```text
schema validity
```

de:

```text
evidence quality
```


In [ ]:
schema_errors = []

for item in BASE_EVIDENCE:
    try:
        validate(instance=item, schema=EVIDENCE_SCHEMA)
    except ValidationError as exc:
        schema_errors.append({
            "evidence_id": item.get("evidence_id"),
            "error": exc.message,
        })

if schema_errors:
    display(pd.DataFrame(schema_errors))
else:
    print("Evidence pack válido según el schema.")


# 8. Integridad del dossier

Calculamos un hash SHA-256 del evidence pack serializado de forma canónica.

El hash no demuestra que la evidencia sea correcta.

Sí permite detectar que el paquete evaluado **ha cambiado**.

En un entorno real podríamos:

- firmar artefactos;
- almacenar hashes en un registro inmutable;
- conservar versiones;
- registrar quién produjo cada evidencia.


In [ ]:
def canonical_json(data):
    return json.dumps(
        data,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

def sha256_of(data):
    return hashlib.sha256(
        canonical_json(data).encode("utf-8")
    ).hexdigest()

BASE_EVIDENCE_HASH = sha256_of(BASE_EVIDENCE)

print("SHA-256 evidence pack:")
print(BASE_EVIDENCE_HASH)


# 9. Parámetros de gobierno

Un control necesita un criterio explícito.

Si no declaramos los límites antes de mirar los resultados, corremos el riesgo de moverlos hasta conseguir un resultado conveniente.

Los siguientes parámetros representan **risk appetite y criterios internos de Velora para este ejercicio**.

No son thresholds universales.


In [ ]:
MIN_DEFAULT_RECALL = 0.50 #@param {type:"slider", min:0.20, max:0.80, step:0.05}
MIN_DEFAULT_PRECISION = 0.30 #@param {type:"slider", min:0.10, max:0.80, step:0.05}

MAX_DP_DIFFERENCE = 0.10 #@param {type:"slider", min:0.02, max:0.30, step:0.01}
MAX_EO_DIFFERENCE = 0.10 #@param {type:"slider", min:0.02, max:0.30, step:0.01}

MAX_EPSILON = 5.0 #@param {type:"slider", min:1.0, max:10.0, step:0.5}

MIN_RAG_HIT_AT_K = 0.90 #@param {type:"slider", min:0.50, max:1.00, step:0.05}
MIN_RAG_MRR = 0.80 #@param {type:"slider", min:0.50, max:1.00, step:0.05}

print("Criterios internos cargados.")


# 10. Escenario de despliegue

El sistema parte de una configuración que **no debería aprobarse automáticamente**.

Podemos aplicar remediaciones:

- seleccionar otro threshold;
- activar fairness mitigation;
- completar runbook de incidentes;
- definir thresholds de monitoring.

El objetivo es comprobar cómo cambia la decisión y qué trade-offs aparecen.


In [ ]:
DECISION_THRESHOLD = 0.5 #@param [0.2, 0.3, 0.4, 0.5, 0.6] {type:"raw"}
USE_FAIRNESS_MITIGATION = False #@param {type:"boolean"}

INCIDENT_RUNBOOK_READY = False #@param {type:"boolean"}
ROLLBACK_PROCEDURE_READY = False #@param {type:"boolean"}

MONITORING_THRESHOLDS_DEFINED = False #@param {type:"boolean"}

print("Threshold:", DECISION_THRESHOLD)
print("Fairness mitigation:", USE_FAIRNESS_MITIGATION)
print("Incident runbook:", INCIDENT_RUNBOOK_READY)
print("Rollback:", ROLLBACK_PROCEDURE_READY)
print("Monitoring thresholds:", MONITORING_THRESHOLDS_DEFINED)


# 11. Construir el evidence pack efectivo

Las remediaciones no sobrescriben el historial.

Creamos una nueva versión efectiva del dossier.

Esto simula una práctica importante de gobierno:

> **no borrar la evidencia anterior; crear una nueva versión con trazabilidad del cambio.**


In [ ]:
def build_effective_evidence():
    bundle = deepcopy(BASE_EVIDENCE)

    by_id = {
        item["evidence_id"]: item
        for item in bundle
    }

    selected_metrics = THRESHOLD_OPTIONS[
        float(DECISION_THRESHOLD)
    ]

    by_id["E-P01-PERF"]["payload"].update({
        "selected_threshold": float(DECISION_THRESHOLD),
        **selected_metrics,
    })

    fairness_id = (
        "E-P02-FAIR-MIT"
        if USE_FAIRNESS_MITIGATION
        else "E-P02-FAIR-BASE"
    )

    by_id["E-P02-FAIR-BASE"]["payload"]["selected_for_decision"] = (
        fairness_id == "E-P02-FAIR-BASE"
    )
    by_id["E-P02-FAIR-MIT"]["payload"]["selected_for_decision"] = (
        fairness_id == "E-P02-FAIR-MIT"
    )

    by_id["E-OPS-INCIDENT"]["payload"].update({
        "ai_incident_runbook": INCIDENT_RUNBOOK_READY,
        "rollback_procedure": ROLLBACK_PROCEDURE_READY,
    })

    by_id["E-OPS-MONITOR"]["payload"].update({
        "performance_drift_thresholds": MONITORING_THRESHOLDS_DEFINED,
        "fairness_drift_thresholds": MONITORING_THRESHOLDS_DEFINED,
    })

    return bundle, fairness_id

EFFECTIVE_EVIDENCE, SELECTED_FAIRNESS_EVIDENCE = (
    build_effective_evidence()
)

EFFECTIVE_EVIDENCE_HASH = sha256_of(EFFECTIVE_EVIDENCE)

print("Base hash     :", BASE_EVIDENCE_HASH[:20], "...")
print("Effective hash:", EFFECTIVE_EVIDENCE_HASH[:20], "...")
print(
    "Pack modificado:",
    BASE_EVIDENCE_HASH != EFFECTIVE_EVIDENCE_HASH,
)


# 12. Catálogo de controles

Cada control contiene:

- dominio;
- referencia conceptual;
- evidencias requeridas;
- criticidad;
- si es blocker;
- regla;
- remediación.

No copiamos cláusulas regulatorias.

Usamos controles internos y los mapeamos a temas de referencia para mejorar trazabilidad.


In [ ]:
CONTROL_CATALOG = [
    {
        "control_id": "GOV-01",
        "domain": "Governance",
        "nist": "GOVERN",
        "eu_theme": "Documentation / accountability",
        "title": "Ownership e intended use",
        "evidence_ids": ["E-GOV-OWNERSHIP"],
        "severity": "HIGH",
        "blocker": True,
        "remediation": "Asignar owners y documentar intended/out-of-scope use.",
    },
    {
        "control_id": "PERF-01",
        "domain": "Performance",
        "nist": "MEASURE",
        "eu_theme": "Accuracy / risk management",
        "title": "Recall y precision mínimos",
        "evidence_ids": ["E-P01-PERF"],
        "severity": "CRITICAL",
        "blocker": True,
        "remediation": "Revisar threshold/modelo y documentar trade-off FP/FN.",
    },
    {
        "control_id": "FAIR-01",
        "domain": "Fairness",
        "nist": "MEASURE",
        "eu_theme": "Data / discriminatory outcomes",
        "title": "Diferencias de fairness dentro de tolerancia",
        "evidence_ids": [
            "E-P02-FAIR-BASE",
            "E-P02-FAIR-MIT",
        ],
        "severity": "CRITICAL",
        "blocker": True,
        "remediation": "Aplicar/ajustar mitigación y reevaluar utilidad y grupos.",
    },
    {
        "control_id": "PRIV-01",
        "domain": "Privacy",
        "nist": "MEASURE",
        "eu_theme": "Risk management",
        "title": "Privacy budget documentado y dentro del límite",
        "evidence_ids": ["E-P03-PRIV"],
        "severity": "HIGH",
        "blocker": True,
        "remediation": "Revisar epsilon, delta, unidad de privacidad y composición.",
    },
    {
        "control_id": "XAI-01",
        "domain": "Explainability",
        "nist": "MEASURE",
        "eu_theme": "Transparency / human oversight",
        "title": "Explicaciones evaluadas y límites documentados",
        "evidence_ids": ["E-P04-XAI"],
        "severity": "MEDIUM",
        "blocker": False,
        "remediation": "Completar fidelidad, estabilidad y restricciones de contrafactuales.",
    },
    {
        "control_id": "RAG-01",
        "domain": "GenAI",
        "nist": "MEASURE",
        "eu_theme": "Accuracy / robustness",
        "title": "Retrieval, grounding, citas y abstención",
        "evidence_ids": ["E-P05-RAG"],
        "severity": "HIGH",
        "blocker": True,
        "remediation": "Mejorar retrieval, citas, abstención o validación de contexto.",
    },
    {
        "control_id": "AGT-01",
        "domain": "Agents",
        "nist": "MANAGE",
        "eu_theme": "Human oversight / cybersecurity",
        "title": "Least privilege y approval enforcement",
        "evidence_ids": ["E-P06-AGENT"],
        "severity": "CRITICAL",
        "blocker": True,
        "remediation": "Limitar tools y mover autorización crítica al backend.",
    },
    {
        "control_id": "OPS-01",
        "domain": "Operations",
        "nist": "MANAGE",
        "eu_theme": "Post-deployment monitoring",
        "title": "Monitoring con thresholds operativos",
        "evidence_ids": ["E-OPS-MONITOR"],
        "severity": "HIGH",
        "blocker": False,
        "remediation": "Definir thresholds, owner, alertas y respuesta ante drift.",
    },
    {
        "control_id": "OPS-02",
        "domain": "Operations",
        "nist": "MANAGE",
        "eu_theme": "Risk management / incidents",
        "title": "Incident response y rollback",
        "evidence_ids": ["E-OPS-INCIDENT"],
        "severity": "CRITICAL",
        "blocker": True,
        "remediation": "Completar runbook de incidentes y procedimiento de rollback.",
    },
]

controls_df = pd.DataFrame(CONTROL_CATALOG)

display(
    controls_df[
        [
            "control_id",
            "domain",
            "nist",
            "eu_theme",
            "severity",
            "blocker",
            "title",
        ]
    ]
)


# 13. Cobertura control → evidencia

Antes de evaluar valores comprobamos que cada control referencia evidencias existentes.

Un control sin evidencia no debería aparecer como `PASS`.

Debe aparecer como:

```text
NO_EVIDENCE
```

Esta distinción evita convertir “no hemos encontrado un problema” en “hemos demostrado que no existe”.


In [ ]:
def evidence_index(bundle):
    return {
        item["evidence_id"]: item
        for item in bundle
    }

EVIDENCE_INDEX = evidence_index(EFFECTIVE_EVIDENCE)

coverage_rows = []

for control in CONTROL_CATALOG:
    missing = [
        evidence_id
        for evidence_id in control["evidence_ids"]
        if evidence_id not in EVIDENCE_INDEX
    ]

    coverage_rows.append({
        "control_id": control["control_id"],
        "required_evidence": ", ".join(control["evidence_ids"]),
        "missing_evidence": ", ".join(missing),
        "coverage": "COMPLETE" if not missing else "MISSING",
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)


# 14. Reglas de evaluación

Las reglas se implementan fuera del modelo.

Esto convierte el gate en una decisión:

- explícita;
- reproducible;
- auditable.

No utilizamos un LLM para decidir si un control pasa.

Un LLM podría ayudar a redactar findings, pero el criterio crítico debe ser verificable.


In [ ]:
def payload(evidence_id):
    return EVIDENCE_INDEX[evidence_id]["payload"]

def evaluate_control(control_id):
    if control_id == "GOV-01":
        p = payload("E-GOV-OWNERSHIP")
        required = [
            "business_owner_assigned",
            "technical_owner_assigned",
            "risk_owner_assigned",
            "intended_use_documented",
            "out_of_scope_documented",
        ]
        ok = all(p.get(k) is True for k in required)
        return (
            "PASS" if ok else "FAIL",
            f"Campos completos: {sum(bool(p.get(k)) for k in required)}/{len(required)}",
        )

    if control_id == "PERF-01":
        p = payload("E-P01-PERF")
        recall_ok = p["recall"] >= MIN_DEFAULT_RECALL
        precision_ok = p["precision"] >= MIN_DEFAULT_PRECISION
        ok = recall_ok and precision_ok
        return (
            "PASS" if ok else "FAIL",
            (
                f"threshold={p['selected_threshold']:.1f}; "
                f"recall={p['recall']:.3f} "
                f"(min {MIN_DEFAULT_RECALL:.2f}); "
                f"precision={p['precision']:.3f} "
                f"(min {MIN_DEFAULT_PRECISION:.2f})"
            ),
        )

    if control_id == "FAIR-01":
        p = payload(SELECTED_FAIRNESS_EVIDENCE)
        dp_ok = (
            abs(p["demographic_parity_difference"])
            <= MAX_DP_DIFFERENCE
        )
        eo_ok = (
            abs(p["equal_opportunity_difference"])
            <= MAX_EO_DIFFERENCE
        )
        ok = dp_ok and eo_ok
        return (
            "PASS" if ok else "FAIL",
            (
                f"source={SELECTED_FAIRNESS_EVIDENCE}; "
                f"DP diff={p['demographic_parity_difference']:.3f} "
                f"(max {MAX_DP_DIFFERENCE:.2f}); "
                f"EO diff={p['equal_opportunity_difference']:.3f} "
                f"(max {MAX_EO_DIFFERENCE:.2f})"
            ),
        )

    if control_id == "PRIV-01":
        p = payload("E-P03-PRIV")
        ok = (
            p["epsilon"] <= MAX_EPSILON
            and p["privacy_unit_documented"]
            and p["composition_tracked"]
        )
        return (
            "PASS" if ok else "FAIL",
            (
                f"epsilon={p['epsilon']}; max={MAX_EPSILON}; "
                f"privacy_unit={p['privacy_unit_documented']}; "
                f"composition={p['composition_tracked']}"
            ),
        )

    if control_id == "XAI-01":
        p = payload("E-P04-XAI")
        required = [
            "global_shap",
            "local_shap",
            "lime_fidelity_checked",
            "lime_stability_checked",
            "counterfactual_constraints",
            "causal_limit_documented",
        ]
        score = sum(bool(p.get(k)) for k in required)
        status = "PASS" if score == len(required) else "CONDITIONAL"
        return (
            status,
            f"Evidencias XAI completas: {score}/{len(required)}",
        )

    if control_id == "RAG-01":
        p = payload("E-P05-RAG")
        ok = all([
            p["retrieval_evaluated"],
            p["hit_at_k"] >= MIN_RAG_HIT_AT_K,
            p["mrr"] >= MIN_RAG_MRR,
            p["invalid_citations"] == 0,
            p["unsupported_question_abstained"],
            p["indirect_injection_tested"],
            p["injection_control_effective"],
        ])
        return (
            "PASS" if ok else "FAIL",
            (
                f"Hit@K={p['hit_at_k']:.3f}; "
                f"MRR={p['mrr']:.3f}; "
                f"invalid_citations={p['invalid_citations']}; "
                f"abstention={p['unsupported_question_abstained']}; "
                f"injection_control={p['injection_control_effective']}"
            ),
        )

    if control_id == "AGT-01":
        p = payload("E-P06-AGENT")
        required = [
            "least_privilege_scope",
            "tool_calls_structured",
            "backend_enforces_approval",
            "approval_outside_agent",
            "approval_single_use",
            "replay_denied",
            "tool_response_injection_tested",
            "response_id_logged",
            "call_id_logged",
        ]
        ok = all(p.get(k) is True for k in required)
        return (
            "PASS" if ok else "FAIL",
            f"Controles agénticos: {sum(bool(p.get(k)) for k in required)}/{len(required)}",
        )

    if control_id == "OPS-01":
        p = payload("E-OPS-MONITOR")
        core = all([
            p["monitoring_owner"],
            p["rag_quality_monitoring"],
            p["agent_denial_rate_monitoring"],
        ])
        thresholds = all([
            p["performance_drift_thresholds"],
            p["fairness_drift_thresholds"],
        ])

        if core and thresholds:
            status = "PASS"
        elif core:
            status = "CONDITIONAL"
        else:
            status = "FAIL"

        return (
            status,
            (
                f"owner={p['monitoring_owner']}; "
                f"performance_thresholds={p['performance_drift_thresholds']}; "
                f"fairness_thresholds={p['fairness_drift_thresholds']}"
            ),
        )

    if control_id == "OPS-02":
        p = payload("E-OPS-INCIDENT")
        ok = (
            p["ai_incident_runbook"]
            and p["escalation_owner"]
            and p["rollback_procedure"]
        )
        return (
            "PASS" if ok else "FAIL",
            (
                f"runbook={p['ai_incident_runbook']}; "
                f"owner={p['escalation_owner']}; "
                f"rollback={p['rollback_procedure']}"
            ),
        )

    return "NO_EVIDENCE", "No existe regla implementada."


# 15. Ejecutar el control gate

Para cada control obtenemos:

- estado;
- evidencia;
- observación;
- criticidad;
- blocker;
- remediación.

Esta tabla constituye la **control matrix** del audit gate.


In [ ]:
evaluation_rows = []

for control in CONTROL_CATALOG:
    missing = [
        evidence_id
        for evidence_id in control["evidence_ids"]
        if evidence_id not in EVIDENCE_INDEX
    ]

    if missing:
        status = "NO_EVIDENCE"
        observation = "Falta: " + ", ".join(missing)
    else:
        status, observation = evaluate_control(
            control["control_id"]
        )

    evaluation_rows.append({
        **control,
        "status": status,
        "observation": observation,
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(
    evaluation_df[
        [
            "control_id",
            "domain",
            "severity",
            "blocker",
            "status",
            "observation",
        ]
    ]
)


# 16. Resumen visual del gate

Contamos estados.

Un dashboard no sustituye el detalle, pero permite detectar rápidamente dónde concentrar revisión.


In [ ]:
status_summary = (
    evaluation_df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="controls")
)

display(status_summary)


# 17. Generar findings

Un finding debe contener algo más que:

> “control fallado”.

Incluimos:

- identificador;
- control;
- riesgo;
- evidencia;
- observación;
- remediación;
- owner.

Los controles `PASS` no generan finding.


In [ ]:
OWNER_BY_DOMAIN = {
    "Governance": "Enterprise Risk",
    "Performance": "Model Risk",
    "Fairness": "Responsible AI",
    "Privacy": "Privacy Office",
    "Explainability": "Model Risk",
    "GenAI": "AI Platform",
    "Agents": "AI Platform + Security",
    "Operations": "AI Operations",
}

findings = []

for _, row in evaluation_df.iterrows():
    if row["status"] == "PASS":
        continue

    findings.append({
        "finding_id": f"F-{len(findings)+1:03d}",
        "control_id": row["control_id"],
        "domain": row["domain"],
        "severity": row["severity"],
        "status": row["status"],
        "observation": row["observation"],
        "evidence_ids": ", ".join(row["evidence_ids"]),
        "remediation": row["remediation"],
        "owner": OWNER_BY_DOMAIN[row["domain"]],
    })

findings_df = pd.DataFrame(findings)

display(findings_df)


# 18. Risk register

Convertimos findings en riesgos priorizables.

Para el ejercicio usamos una matriz simple:

```text
risk score = likelihood × impact
```

Escala 1–5.

Esto es un modelo interno del laboratorio, no una escala regulatoria universal.


In [ ]:
RISK_PARAMETERS = {
    "GOV-01": (2, 4),
    "PERF-01": (4, 5),
    "FAIR-01": (4, 5),
    "PRIV-01": (3, 5),
    "XAI-01": (2, 3),
    "RAG-01": (3, 4),
    "AGT-01": (4, 5),
    "OPS-01": (3, 4),
    "OPS-02": (4, 5),
}

def risk_level(score):
    if score >= 17:
        return "CRITICAL"
    if score >= 10:
        return "HIGH"
    if score >= 5:
        return "MEDIUM"
    return "LOW"

risk_rows = []

for finding in findings:
    likelihood, impact = RISK_PARAMETERS[
        finding["control_id"]
    ]
    score = likelihood * impact

    risk_rows.append({
        "risk_id": f"R-{len(risk_rows)+1:03d}",
        "finding_id": finding["finding_id"],
        "control_id": finding["control_id"],
        "likelihood": likelihood,
        "impact": impact,
        "risk_score": score,
        "risk_level": risk_level(score),
        "owner": finding["owner"],
        "treatment": (
            "MITIGATE"
            if finding["status"] == "FAIL"
            else "MITIGATE / ACCEPT WITH CONDITIONS"
        ),
    })

risk_df = pd.DataFrame(risk_rows)

display(
    risk_df.sort_values(
        ["risk_score"],
        ascending=False,
    )
)


# 19. Decisión de despliegue

Aplicamos una regla explícita:

### BLOCKED

Si existe:

- un blocker en `FAIL`;
- un blocker sin evidencia.

### CONDITIONAL

Si no hay blockers fallados, pero quedan controles `CONDITIONAL` o findings no bloqueantes.

### APPROVED

Si todos los controles pasan.

La decisión no la genera un LLM.


In [ ]:
def deployment_decision(evaluation_df):
    blocking = evaluation_df[
        (evaluation_df["blocker"] == True)
        & (evaluation_df["status"].isin(
            ["FAIL", "NO_EVIDENCE"]
        ))
    ]

    non_pass = evaluation_df[
        evaluation_df["status"] != "PASS"
    ]

    if len(blocking) > 0:
        return "BLOCKED", blocking["control_id"].tolist()

    if len(non_pass) > 0:
        return "CONDITIONAL", non_pass["control_id"].tolist()

    return "APPROVED", []

DECISION, DECISION_REASONS = deployment_decision(
    evaluation_df
)

print("DECISIÓN:", DECISION)
print("Controles que condicionan/bloquean:", DECISION_REASONS)


# 20. Interpretar la primera decisión

Con los valores iniciales esperamos un resultado `BLOCKED`.

Los motivos principales deberían ser:

- recall insuficiente a threshold 0.5;
- fairness fuera de tolerancia;
- ausencia de runbook/rollback.

Esto demuestra una idea importante:

> Un sistema puede tener RAG, agentes, XAI y privacy controls correctos y seguir sin estar listo para desplegarse.

La gobernanza es end-to-end.


# 21. Remediación 1 — threshold

Prueba:

```python
DECISION_THRESHOLD = 0.2
```

y vuelve a ejecutar desde la construcción del evidence pack efectivo.

En P01 sabemos que:

- aumenta recall;
- aumenta falsos positivos.

El gate puede pasar el control de recall, pero eso **no elimina el trade-off**.

Gobierno no significa maximizar una única métrica.


# 22. Remediación 2 — fairness

Activa:

```python
USE_FAIRNESS_MITIGATION = True
```

El gate utilizará el snapshot mitigado.

Después verifica:

- DP difference;
- EO difference;
- cualquier impacto de utilidad que deba revisarse.

No debemos interpretar “fairness PASS” como “sistema justo en sentido absoluto”.

Significa:

> satisface los criterios de fairness definidos para este gate y estas métricas.


# 23. Remediación 3 — incident response

Activa:

```python
INCIDENT_RUNBOOK_READY = True
ROLLBACK_PROCEDURE_READY = True
```

Esto convierte un control organizativo en evidencia verificable.

La práctica muestra que la confiabilidad no depende solo de código y métricas.

También depende de:

- quién responde;
- cómo se escala;
- cómo se revierte una versión;
- qué hacer cuando algo falla en producción.


# 24. Remediación 4 — monitoring

Activa:

```python
MONITORING_THRESHOLDS_DEFINED = True
```

No basta con decir:

> “monitorizamos el modelo”.

Debemos definir qué evento provoca acción.

Ejemplos:

- caída de recall;
- aumento de fairness gap;
- degradación de retrieval;
- subida de tool denials;
- cambios anómalos en side effects.

Monitoring sin thresholds puede generar datos sin capacidad de respuesta.


# 25. Ejecutar un escenario remediado

Esta celda activa una configuración que debería eliminar los blockers.

Puedes modificarla para explorar estados `CONDITIONAL` y `APPROVED`.


In [ ]:
# Configuración de referencia remediada
DECISION_THRESHOLD = 0.2
USE_FAIRNESS_MITIGATION = True
INCIDENT_RUNBOOK_READY = True
ROLLBACK_PROCEDURE_READY = True
MONITORING_THRESHOLDS_DEFINED = False

EFFECTIVE_EVIDENCE, SELECTED_FAIRNESS_EVIDENCE = (
    build_effective_evidence()
)
EVIDENCE_INDEX = evidence_index(EFFECTIVE_EVIDENCE)

evaluation_rows = []

for control in CONTROL_CATALOG:
    missing = [
        evidence_id
        for evidence_id in control["evidence_ids"]
        if evidence_id not in EVIDENCE_INDEX
    ]

    if missing:
        status = "NO_EVIDENCE"
        observation = "Falta: " + ", ".join(missing)
    else:
        status, observation = evaluate_control(
            control["control_id"]
        )

    evaluation_rows.append({
        **control,
        "status": status,
        "observation": observation,
    })

evaluation_df = pd.DataFrame(evaluation_rows)

DECISION, DECISION_REASONS = deployment_decision(
    evaluation_df
)

print("DECISIÓN REMEDIADA:", DECISION)
print("Controles pendientes:", DECISION_REASONS)

display(
    evaluation_df[
        [
            "control_id",
            "status",
            "observation",
        ]
    ]
)


## 26. ¿Por qué puede quedar `CONDITIONAL`?

Aunque los blockers estén resueltos, si `OPS-01` mantiene:

```text
performance_drift_thresholds = False
fairness_drift_thresholds = False
```

el sistema puede quedar:

```text
CONDITIONAL
```

Eso permite distinguir:

### Bloqueo

No debería desplegarse.

### Condicional

Puede avanzar solo bajo condiciones, plazo y owner definidos.

### Aprobación

El gate no detecta findings abiertos según el catálogo actual.

**APPROVED no significa “sin riesgo”.**

Significa “riesgo aceptado dentro de los criterios definidos”.


# 27. Completar monitoring y reevaluar

Activa el último control pendiente.


In [ ]:
MONITORING_THRESHOLDS_DEFINED = True

EFFECTIVE_EVIDENCE, SELECTED_FAIRNESS_EVIDENCE = (
    build_effective_evidence()
)
EVIDENCE_INDEX = evidence_index(EFFECTIVE_EVIDENCE)

evaluation_rows = []

for control in CONTROL_CATALOG:
    status, observation = evaluate_control(
        control["control_id"]
    )

    evaluation_rows.append({
        **control,
        "status": status,
        "observation": observation,
    })

evaluation_df = pd.DataFrame(evaluation_rows)

DECISION, DECISION_REASONS = deployment_decision(
    evaluation_df
)

print("DECISIÓN FINAL:", DECISION)
print("Pendientes:", DECISION_REASONS)

display(
    evaluation_df[
        [
            "control_id",
            "domain",
            "status",
        ]
    ]
)


# 28. Traceability matrix

Una auditoría debe poder responder:

> ¿Qué evidencia sostiene cada control?

Generamos una matriz explícita control → evidencia.

Esto facilita:

- revisión;
- actualización;
- auditoría externa;
- detectar evidencias huérfanas;
- detectar controles sin soporte.


In [ ]:
trace_rows = []

for control in CONTROL_CATALOG:
    for evidence_id in control["evidence_ids"]:
        item = EVIDENCE_INDEX.get(evidence_id)

        trace_rows.append({
            "control_id": control["control_id"],
            "control_title": control["title"],
            "evidence_id": evidence_id,
            "evidence_source": (
                item["source"]
                if item else "MISSING"
            ),
            "evidence_version": (
                item["version"]
                if item else "MISSING"
            ),
            "status": evaluation_df.loc[
                evaluation_df["control_id"]
                == control["control_id"],
                "status",
            ].iloc[0],
        })

traceability_df = pd.DataFrame(trace_rows)

display(traceability_df)


# 29. Cambio de evidencia y re-auditoría

Una aprobación no debería ser eterna.

Si cambia:

- modelo;
- threshold;
- dataset;
- prompt;
- vector store;
- tool;
- permisos;
- policy;
- proveedor;

puede invalidarse parte del dossier.

Esto es **change impact analysis**.

No todos los cambios obligan a repetir todas las pruebas, pero debemos saber qué controles se ven afectados.


In [ ]:
CHANGE_IMPACT = {
    "threshold": [
        "PERF-01",
        "FAIR-01",
    ],
    "training_data": [
        "PERF-01",
        "FAIR-01",
        "PRIV-01",
        "XAI-01",
    ],
    "embedding_model": [
        "RAG-01",
    ],
    "rag_corpus": [
        "RAG-01",
    ],
    "llm_model": [
        "RAG-01",
        "AGT-01",
    ],
    "tool_schema": [
        "AGT-01",
    ],
    "approval_policy": [
        "AGT-01",
        "OPS-02",
    ],
}

CHANGED_COMPONENT = "llm_model" #@param ["threshold", "training_data", "embedding_model", "rag_corpus", "llm_model", "tool_schema", "approval_policy"]

print("Cambio:", CHANGED_COMPONENT)
print(
    "Controles que deben reevaluarse:",
    CHANGE_IMPACT[CHANGED_COMPONENT],
)


# 30. Generar informe de auditoría

Finalmente convertimos la evidencia estructurada en un artefacto legible.

El informe incluirá:

- contexto;
- hashes;
- decisión;
- control matrix;
- findings;
- riesgos;
- trazabilidad;
- controles afectados por cambios.

La generación del informe es determinista.

Un LLM podría ayudar a redactar un executive summary, pero no debería alterar los resultados del gate.


In [ ]:
def markdown_table(df):
    if df is None or len(df) == 0:
        return "_Sin registros._"

    return df.to_markdown(index=False)

final_findings = evaluation_df[
    evaluation_df["status"] != "PASS"
].copy()

report = f'''# Velora AI — Informe de auditoría técnica

**System ID:** {SYSTEM_CONTEXT["system_id"]}  
**Version:** {SYSTEM_CONTEXT["version"]}  
**Stage:** {SYSTEM_CONTEXT["stage"]}  
**Audit date:** {AUDIT_TIMESTAMP}  

## Decisión

**{DECISION}**

Controles pendientes: {", ".join(DECISION_REASONS) if DECISION_REASONS else "Ninguno"}

## Integridad del dossier

- Base evidence SHA-256: `{BASE_EVIDENCE_HASH}`
- Effective evidence SHA-256: `{sha256_of(EFFECTIVE_EVIDENCE)}`

## Contexto

- Intended use: {SYSTEM_CONTEXT["intended_use"]}
- Business owner: {SYSTEM_CONTEXT["business_owner"]}
- Technical owner: {SYSTEM_CONTEXT["technical_owner"]}
- Risk owner: {SYSTEM_CONTEXT["risk_owner"]}

## Control matrix

{markdown_table(evaluation_df[[
    "control_id",
    "domain",
    "severity",
    "blocker",
    "status",
    "observation",
]])}

## Findings abiertos

{markdown_table(final_findings[[
    "control_id",
    "domain",
    "severity",
    "status",
    "observation",
    "remediation",
]])}

## Trazabilidad

{markdown_table(traceability_df)}

## Change impact seleccionado

Cambio: **{CHANGED_COMPONENT}**

Reevaluar:

{chr(10).join(f"- {x}" for x in CHANGE_IMPACT[CHANGED_COMPONENT])}

## Nota

Este informe es un artefacto técnico de formación.
No constituye certificación, opinión legal ni declaración de cumplimiento regulatorio.
'''

REPORT_PATH = "/content/P07_Velora_Audit_Report.md"

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(report)

print(report[:5000])
print("\nInforme generado en:", REPORT_PATH)


# 31. Prueba opcional de integridad

Podemos modificar una copia del dossier después de calcular el hash.

El hash cambiará.

Esto demuestra detección de alteración, no autenticidad del origen.

Para autenticidad necesitaríamos, por ejemplo:

- firma digital;
- identidad del productor;
- cadena de confianza;
- repositorio protegido.


In [ ]:
RUN_TAMPER_DEMO = False #@param {type:"boolean"}

if RUN_TAMPER_DEMO:
    tampered = deepcopy(EFFECTIVE_EVIDENCE)

    tampered[0]["payload"]["roc_auc"] = 0.999

    original_hash = sha256_of(EFFECTIVE_EVIDENCE)
    tampered_hash = sha256_of(tampered)

    print("Hash original :", original_hash)
    print("Hash alterado :", tampered_hash)
    print("Coinciden     :", original_hash == tampered_hash)
else:
    print("Demo de alteración desactivada.")


# 32. Qué NO demuestra un `APPROVED`

Si el gate devuelve:

```text
APPROVED
```

no hemos demostrado que:

- el sistema sea seguro frente a todos los ataques;
- no exista discriminación;
- el LLM no pueda alucinar;
- no haya futuros incidentes;
- exista cumplimiento legal completo;
- la organización esté certificada.

Solo hemos demostrado que:

> **las evidencias disponibles satisfacen el catálogo y los límites definidos para esta revisión.**

La gobernanza debe continuar durante todo el lifecycle.


# Conclusiones de P07

- La gobernanza necesita **evidencia**, no solo principios.
- Un control sin evidencia no debería considerarse superado.
- Los thresholds deben declararse y justificarse antes de interpretar resultados.
- Performance, fairness, privacy, XAI, RAG y agentes deben evaluarse como partes del mismo sistema.
- Los controles críticos deben ser reproducibles y verificables.
- Los findings deben tener owner y remediación.
- Un risk register convierte problemas técnicos en decisiones priorizables.
- `BLOCKED`, `CONDITIONAL` y `APPROVED` representan decisiones distintas.
- `APPROVED` no significa riesgo cero.
- La integridad del evidence pack permite detectar cambios.
- La trazabilidad control → evidencia permite reconstruir por qué se tomó una decisión.
- Un cambio de modelo, threshold, corpus o tools puede obligar a reevaluar controles concretos.
- La auditoría no termina en pre-producción: monitoring, incident response y change management forman parte del lifecycle.

El hilo completo del curso queda así:

```text
P01  ¿Funciona el modelo?
 ↓
P02  ¿Se comporta aceptablemente entre grupos?
 ↓
P03  ¿Qué garantía de privacidad tenemos?
 ↓
P04  ¿Podemos explicar predicciones?
 ↓
P05  ¿La respuesta generativa está grounded?
 ↓
P06  ¿Puede el agente actuar con autoridad controlada?
 ↓
P07  ¿Tenemos evidencia suficiente para gobernar y auditar el sistema?
```
